In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Tools for building the machine learning model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV, train_test_split
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# 1. Load the data
df = pd.read_csv('train.csv')

In [3]:
# 2. Explore the data to see what is missing
df.info()
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
# 3. Clean the data
# Remove 'Cabin' because it has too many missing values
df.drop('Cabin', axis=1, inplace=True)

# Remove columns that aren't useful for prediction
df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)

# NOTE: missing 'Age' and 'Embarked' values are NOT filled here. They are imputed inside the
# preprocessing pipeline below, so medians/modes are learned from training data only.


In [5]:
# 4. Feature Engineering
# Create 'FamilySize' by adding siblings, parents, and the passenger themselves
X = df.drop('Survived', axis=1)
y = df['Survived']
X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
X.drop(['SibSp', 'Parch'], axis=1, inplace=True)

# 5. Split and Preprocess the data


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['Age', 'Fare', 'FamilySize']
cat_cols = ['Pclass', 'Sex', 'Embarked']

# Impute (median for numbers, most frequent for categories), then scale / encode.
# Imputation values are learned from the training data only (and from each training fold in CV).
preprocessor = ColumnTransformer(transformers=[
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat_cols)
])


In [7]:
# 6. Build the Model Pipeline
# Using SMOTE to balance the data and Random Forest to classify
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('sampler', SMOTE(random_state=42)), 
    ('classifier', RandomForestClassifier(random_state=42))
])

In [8]:
# 7. Fine-tuning the Model
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None]
}

# Find the best version of the model using cross-validation
grid_search = GridSearchCV(pipeline, param_grid, scoring='recall_macro', cv=5)
grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV macro recall:', round(grid_search.best_score_, 3))


Best parameters: {'classifier__max_depth': 5, 'classifier__n_estimators': 50}
Best CV macro recall: 0.806


In [9]:
# 8. Final Evaluation (test set used once)
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))
print('Majority-class baseline accuracy:', round(y_test.value_counts(normalize=True).max(), 3))


              precision    recall  f1-score   support

           0       0.85      0.81      0.83       110
           1       0.72      0.77      0.74        69

    accuracy                           0.79       179
   macro avg       0.78      0.79      0.78       179
weighted avg       0.80      0.79      0.79       179

Majority-class baseline accuracy: 0.615
